# 02 — Мультимодальный анализ: извлечение и описание изображений

Этот ноутбук демонстрирует:
1. Извлечение изображений из PDF
2. Генерацию подписей (OCR + CLIP)
3. Визуализации для отчёта (графики, тепловые карты, PCA)

**Результаты визуализаций сохраняются в `report/img/`**

In [1]:
import os, sys
os.chdir('/app')
sys.path.insert(0, '/app')

import fitz
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Настройки графиков
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

IMG_DIR = 'report/img'
os.makedirs(IMG_DIR, exist_ok=True)

PDF_PATH = 'data/input_pdfs/Курсовая.pdf'
print(f'PDF: {PDF_PATH}')
print(f'Графики → {IMG_DIR}/')

PDF: data/input_pdfs/Курсовая.pdf
Графики → report/img/


---
## 1. Извлечение изображений из PDF

In [2]:
from src.preprocessing.image_extractor import ImageExtractor

extractor = ImageExtractor(min_width=50, min_height=50)
extracted_images = extractor.extract_from_pdf(
    PDF_PATH,
    output_dir='data/images/kursovaya'
)

summary = extractor.get_summary(extracted_images)
print(f"\n=== Результаты извлечения ===")
print(f"Всего изображений: {summary['total']}")
print(f"Страниц с изображениями: {summary['pages_with_images']}")
print(f"Страницы: {summary['pages_list']}")
print(f"Средний размер: {summary['avg_size'][0]}x{summary['avg_size'][1]} px")

# Показать превью
if extracted_images:
    n = min(len(extracted_images), 6)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()
    for i in range(n):
        img = extracted_images[i]
        axes[i].imshow(img.image)
        axes[i].set_title(f'Стр. {img.page_num+1} ({img.width}x{img.height})', fontsize=10)
        axes[i].axis('off')
    for i in range(n, len(axes)):
        axes[i].axis('off')
    plt.suptitle('Извлечённые изображения (превью)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{IMG_DIR}/extracted_images_preview.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nСохранено: {IMG_DIR}/extracted_images_preview.png")
else:
    print('Изображения не найдены в PDF.')


=== Результаты извлечения ===
Всего изображений: 3
Страниц с изображениями: 3
Страницы: [1, 9, 10]
Средний размер: 1263x741 px

Сохранено: report/img/extracted_images_preview.png


---
## 2. Генерация подписей к изображениям

In [3]:
from src.preprocessing.image_captioner import ImageCaptioner

captioner = ImageCaptioner(use_clip=False)  # CLIP опционален

if extracted_images:
    captions = captioner.generate_captions_batch(extracted_images, strategy='auto')
    
    print("=== Подписи к изображениям ===")
    for i, cap in enumerate(captions):
        print(f"\n[{i+1}] Стр. {cap['page_num']+1} | Стратегия: {cap['strategy']}")
        print(f"    Подпись: {cap['caption'][:100]}")
        if cap['ocr_text']:
            print(f"    OCR ({len(cap['ocr_text'])} симв.): {cap['ocr_text'][:80]}...")
else:
    captions = []
    print('Нет изображений для обработки.')

=== Подписи к изображениям ===

[1] Стр. 1 | Стратегия: fallback
    Подпись: Изображение со стр. 1

[2] Стр. 9 | Стратегия: ocr
    Подпись: Снижение 1055-функции при обучении модели Wav2Vec2
a, —e Обучающая выборка
` -@-
22 “- e Валидационн
    OCR (233 симв.): Снижение 1055-функции при обучении модели Wav2Vec2
a, —e Обучающая выборка
` -@-...

[3] Стр. 10 | Стратегия: ocr
    Подпись: Снижение WER по эпохам обучения
0.42
0.40
е
ш
=
а 0.38
в
(5
ac
5
ta 0.36
z
е)
Е
0.34
0.32
1.0 1.5 2.
    OCR (131 симв.): Снижение WER по эпохам обучения
0.42
0.40
е
ш
=
а 0.38
в
(5
ac
5
ta 0.36
z
е)
Е
...


---
## 3. Layout Analysis — полный анализ документа

In [4]:
from src.pipeline.page_analyzer import PageAnalyzer

analyzer = PageAnalyzer()
doc = fitz.open(PDF_PATH)
num_pages = len(doc)

all_blocks = []
blocks_per_page = []

for page_num in range(num_pages):
    page = doc[page_num]
    blocks = analyzer.analyze_page(page, page_num)
    all_blocks.extend(blocks)
    blocks_per_page.append(len(blocks))

doc.close()

print(f"Документ: {os.path.basename(PDF_PATH)}")
print(f"Страниц: {num_pages}")
print(f"Всего блоков: {len(all_blocks)}")

# Подсчёт типов
type_counts = Counter(b.block_type for b in all_blocks)
print(f"\nТипы блоков:")
for bt, count in type_counts.most_common():
    print(f"  {bt}: {count}")

Документ: Курсовая.pdf
Страниц: 12
Всего блоков: 170

Типы блоков:
  paragraph: 94
  list_item: 48
  table: 16
  header: 9
  image: 3


---
## 4. Визуализация: распределение блоков по страницам

Stacked bar chart — количество блоков каждого типа на каждой странице.

In [5]:
# Подготовка данных: тип блока × страница
pages = list(range(1, len(blocks_per_page) + 1))
block_types = sorted(type_counts.keys())

# Матрица: [type][page]
type_page_matrix = {}
for bt in block_types:
    type_page_matrix[bt] = [0] * len(pages)

for b in all_blocks:
    type_page_matrix[b.block_type][b.page_num] += 1

# Stacked bar chart
colors = {
    'paragraph': '#4A90D9',
    'header': '#E74C3C',
    'list_item': '#2ECC71',
    'table': '#F39C12',
    'image': '#9B59B6',
}

fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(pages))

for bt in block_types:
    values = type_page_matrix[bt]
    color = colors.get(bt, '#95A5A6')
    ax.bar(pages, values, bottom=bottom, label=bt.capitalize(), color=color, edgecolor='white', linewidth=0.5)
    bottom += np.array(values)

ax.set_xlabel('Номер страницы', fontsize=13)
ax.set_ylabel('Количество блоков', fontsize=13)
ax.set_title('Распределение блоков по страницам', fontsize=15, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.set_xticks(pages)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/blocks_per_page_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {IMG_DIR}/blocks_per_page_stacked.png")

Сохранено: report/img/blocks_per_page_stacked.png


---
## 5. Визуализация: Layout Analysis на одной странице

Рендер страницы PDF с цветными рамками вокруг найденных блоков.

In [6]:
# Выбираем страницу с наибольшим количеством блоков
best_page = int(np.argmax(blocks_per_page))
print(f"Страница с максимумом блоков: {best_page + 1} ({blocks_per_page[best_page]} блоков)")

doc = fitz.open(PDF_PATH)
page = doc[best_page]

# Рендер страницы
pix = page.get_pixmap(dpi=150)
page_img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)

# Блоки этой страницы
page_blocks = [b for b in all_blocks if b.page_num == best_page]

fig, ax = plt.subplots(figsize=(10, 14))
ax.imshow(page_img)

scale_x = pix.width / page.rect.width
scale_y = pix.height / page.rect.height

for b in page_blocks:
    x0, y0, x1, y1 = b.bbox
    x0 *= scale_x; x1 *= scale_x
    y0 *= scale_y; y1 *= scale_y
    
    color = colors.get(b.block_type, '#95A5A6')
    rect = patches.Rectangle(
        (x0, y0), x1 - x0, y1 - y0,
        linewidth=2, edgecolor=color, facecolor=color, alpha=0.15
    )
    ax.add_patch(rect)
    ax.text(x0, y0 - 3, b.block_type, fontsize=7, color=color, fontweight='bold')

ax.set_title(f'Layout Analysis — стр. {best_page + 1}', fontsize=14, fontweight='bold')
ax.axis('off')

# Легенда
legend_elements = [patches.Patch(facecolor=colors.get(bt, '#95A5A6'), alpha=0.5, label=bt.capitalize()) for bt in block_types]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/layout_analysis_page.png', dpi=150, bbox_inches='tight')
plt.show()
doc.close()
print(f"Сохранено: {IMG_DIR}/layout_analysis_page.png")

Страница с максимумом блоков: 6 (23 блоков)
Сохранено: report/img/layout_analysis_page.png


---
## 6. Chunking + Embeddings

In [7]:
from src.preprocessing.markdown_builder import MarkdownBuilder
from src.rag.chunker import MarkdownChunker
from src.rag.embedder import TextEmbedder

# Markdown
builder = MarkdownBuilder()
markdown_text = builder.build(all_blocks)
print(f"Markdown: {len(markdown_text)} символов")

# Chunking
chunker = MarkdownChunker(max_chunk_size=1000, chunk_overlap=100, min_chunk_size=50)
chunk_objects = chunker.chunk_markdown(markdown_text, source=os.path.basename(PDF_PATH))
# Конвертация Chunk → dict для удобства
chunks = [c.to_dict() for c in chunk_objects]
print(f"Чанков: {len(chunks)}")

# Embeddings
embedder = TextEmbedder()
chunk_texts = [c['text'] for c in chunks]
embeddings = embedder.embed(chunk_texts)
print(f"Embeddings: {embeddings.shape}")

Markdown: 23322 символов
Чанков: 33


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings: (33, 384)


---
## 7. Визуализация: распределение размеров чанков

In [8]:
chunk_sizes = [len(c['text']) for c in chunks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма
ax1.hist(chunk_sizes, bins=20, color='#4A90D9', edgecolor='white', alpha=0.85)
ax1.axvline(np.mean(chunk_sizes), color='#E74C3C', linestyle='--', linewidth=2, label=f'Среднее: {np.mean(chunk_sizes):.0f}')
ax1.axvline(np.median(chunk_sizes), color='#2ECC71', linestyle='--', linewidth=2, label=f'Медиана: {np.median(chunk_sizes):.0f}')
ax1.set_xlabel('Размер чанка (символы)', fontsize=12)
ax1.set_ylabel('Количество', fontsize=12)
ax1.set_title('Распределение размеров чанков', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

# Box plot
bp = ax2.boxplot(chunk_sizes, vert=True, patch_artist=True,
                  boxprops=dict(facecolor='#4A90D9', alpha=0.7),
                  medianprops=dict(color='#E74C3C', linewidth=2))
ax2.set_ylabel('Размер (символы)', fontsize=12)
ax2.set_title('Box plot размеров чанков', fontsize=13, fontweight='bold')
ax2.set_xticklabels(['Чанки'])

# Статистика
stats_text = f'min={min(chunk_sizes)}, max={max(chunk_sizes)}\nμ={np.mean(chunk_sizes):.0f}, σ={np.std(chunk_sizes):.0f}'
ax2.text(0.95, 0.95, stats_text, transform=ax2.transAxes, fontsize=10,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/chunk_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {IMG_DIR}/chunk_size_distribution.png")

Сохранено: report/img/chunk_size_distribution.png


---
## 8. Визуализация: PCA проекция эмбеддингов

In [9]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# PCA: 384d → 2d
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

# Кластеризация (для раскраски)
n_clusters = min(5, len(chunks))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))

cluster_colors = ['#4A90D9', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
for i in range(n_clusters):
    mask = labels == i
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               c=cluster_colors[i], s=80, alpha=0.7,
               label=f'Кластер {i+1} ({mask.sum()} чанков)',
               edgecolors='white', linewidths=0.5)

# Подписи к точкам
for i, c in enumerate(chunks):
    ax.annotate(f'{i+1}', (emb_2d[i, 0], emb_2d[i, 1]),
                fontsize=7, alpha=0.6, ha='center')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} дисперсии)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} дисперсии)', fontsize=12)
ax.set_title('Проекция эмбеддингов чанков (PCA + KMeans)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/embedding_space_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nОбъяснённая дисперсия: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
print(f"Сохранено: {IMG_DIR}/embedding_space_pca.png")


Объяснённая дисперсия: PC1=25.0%, PC2=12.3%
Сохранено: report/img/embedding_space_pca.png


---
## 9. Визуализация: тепловая карта сходства чанков

In [10]:
# Cosine similarity matrix
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normed = embeddings / (norms + 1e-10)
similarity_matrix = normed @ normed.T

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(similarity_matrix, cmap='RdYlBu_r', vmin=0, vmax=1,
               interpolation='nearest')
plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.8)

ax.set_xlabel('Чанк №', fontsize=12)
ax.set_ylabel('Чанк №', fontsize=12)
ax.set_title('Тепловая карта cosine similarity между чанками', fontsize=14, fontweight='bold')

# Метки (если не слишком много чанков)
if len(chunks) <= 40:
    ticks = list(range(len(chunks)))
    tick_labels = [str(i+1) for i in ticks]
    ax.set_xticks(ticks)
    ax.set_xticklabels(tick_labels, fontsize=8)
    ax.set_yticks(ticks)
    ax.set_yticklabels(tick_labels, fontsize=8)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Статистика
upper_tri = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
print(f"\nСтатистика сходства (верхний треугольник):")
print(f"  Среднее: {upper_tri.mean():.3f}")
print(f"  Макс (не диагональ): {upper_tri.max():.3f}")
print(f"  Мин: {upper_tri.min():.3f}")
print(f"Сохранено: {IMG_DIR}/similarity_heatmap.png")


Статистика сходства (верхний треугольник):
  Среднее: 0.561
  Макс (не диагональ): 0.871
  Мин: 0.024
Сохранено: report/img/similarity_heatmap.png


---
## 10. Визуализация: Retrieval — релевантность по запросу

In [11]:
# Демо-запрос
query = "О чём этот документ?"
query_emb = embedder.embed(query)

# Cosine sim запроса со всеми чанками
sims = (normed @ query_emb.T).flatten()

# Сортировка по убыванию
sorted_idx = np.argsort(sims)[::-1]
top_k = 5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Левый: bar chart всех чанков
bar_colors = ['#E74C3C' if i in sorted_idx[:top_k] else '#4A90D9' for i in range(len(sims))]
ax1.bar(range(1, len(sims)+1), sims, color=bar_colors, alpha=0.8, edgecolor='white')
ax1.set_xlabel('Чанк №', fontsize=12)
ax1.set_ylabel('Cosine Similarity', fontsize=12)
ax1.set_title(f'Релевантность чанков к запросу', fontsize=13, fontweight='bold')
ax1.axhline(y=sims[sorted_idx[top_k-1]], color='#E74C3C', linestyle='--', alpha=0.5, label=f'Top-{top_k} порог')
ax1.legend(fontsize=10)

# Правый: top-K чанки с превью
top_labels = []
top_scores = []
for i, idx in enumerate(sorted_idx[:top_k]):
    text = chunks[idx]['text'][:40].replace('\n', ' ')
    top_labels.append(f'#{idx+1}: {text}...')
    top_scores.append(float(sims[idx]))

y_pos = np.arange(top_k)
ax2.barh(y_pos, top_scores, color='#E74C3C', alpha=0.8, edgecolor='white')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(top_labels, fontsize=9)
ax2.set_xlabel('Cosine Similarity', fontsize=12)
ax2.set_title(f'Top-{top_k} чанков', fontsize=13, fontweight='bold')
ax2.invert_yaxis()

for i, v in enumerate(top_scores):
    ax2.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10, fontweight='bold')

plt.suptitle(f'Запрос: "{query}"', fontsize=14, style='italic', y=1.02)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/retrieval_relevance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {IMG_DIR}/retrieval_relevance.png")

Сохранено: report/img/retrieval_relevance.png


---
## 11. Визуализация: архитектура пайплайна (этапы + метрики)

In [12]:
# Pipeline stages summary
stages = [
    ('PDF Input', f'{num_pages} стр.', '#3498DB'),
    ('Layout Analysis', f'{len(all_blocks)} блоков', '#E74C3C'),
    ('Markdown', f'{len(markdown_text)} симв.', '#2ECC71'),
    ('Chunking', f'{len(chunks)} чанков', '#F39C12'),
    ('Embeddings', f'{embeddings.shape[1]}d × {embeddings.shape[0]}', '#9B59B6'),
    ('FAISS Index', f'{embeddings.shape[0]} векторов', '#1ABC9C'),
    ('Retrieval', f'top-{top_k}', '#E67E22'),
    ('GPT API', 'gpt-oss-20b', '#2C3E50'),
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, len(stages))
ax.set_ylim(-1, 2)
ax.axis('off')

for i, (name, value, color) in enumerate(stages):
    x = i + 0.5
    # Блок
    rect = patches.FancyBboxPatch(
        (i + 0.05, 0), 0.9, 1.2,
        boxstyle="round,pad=0.05",
        facecolor=color, alpha=0.85, edgecolor='white', linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x, 0.85, name, ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')
    ax.text(x, 0.35, value, ha='center', va='center', fontsize=9, color='white')
    
    # Стрелка
    if i < len(stages) - 1:
        ax.annotate('', xy=(i + 1.05, 0.6), xytext=(i + 0.95, 0.6),
                    arrowprops=dict(arrowstyle='->', color='#34495E', lw=2))

ax.set_title('Пайплайн обработки документа', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/pipeline_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {IMG_DIR}/pipeline_overview.png")

Сохранено: report/img/pipeline_overview.png


---
## 12. Сводка сохранённых визуализаций

In [13]:
import glob

saved_images = sorted(glob.glob(f'{IMG_DIR}/*.png'))

print(f"=== Все визуализации в {IMG_DIR}/ ===")
print(f"Всего файлов: {len(saved_images)}\n")

for i, path in enumerate(saved_images, 1):
    name = os.path.basename(path)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {i:2d}. {name:<40s} ({size_kb:.0f} KB)")

print(f"\n✅ Все графики готовы для использования в report.tex")
print(f"   Пример: \\includegraphics[width=0.7\\textwidth]{{img/similarity_heatmap.png}}")

=== Все визуализации в report/img/ ===
Всего файлов: 17

   1. block_types_distribution.png             (77 KB)
   2. blocks_per_page_stacked.png              (53 KB)
   3. chunk_distribution.png                   (60 KB)
   4. chunk_size_distribution.png              (77 KB)
   5. embedding_space_pca.png                  (95 KB)
   6. extracted_images_preview.png             (197 KB)
   7. layout_analysis_page.png                 (651 KB)
   8. layout_analysis_page1.png                (66 KB)
   9. mermaid-diagram.png                      (35 KB)
  10. metrics_comparison.png                   (54 KB)
  11. msu_logo.png                             (16 KB)
  12. pipeline_overview.png                    (59 KB)
  13. rag_architecture.png                     (87 KB)
  14. recall_at_k.png                          (69 KB)
  15. retrieval_relevance.png                  (98 KB)
  16. retrieval_scores.png                     (80 KB)
  17. similarity_heatmap.png                   (97 KB)

✅ Все